In [1]:
!pip install rasterio tensorflow pandas numpy scikit-learn tqdm

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.utils import class_weight
from tqdm import tqdm
from google.colab import drive
from scipy.ndimage import distance_transform_edt
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

tf.keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
points_xlsx = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/ls_non_ls_coord.xlsx"

raster_files = [[
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/aspect_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/curvature_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_fault_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_river_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_thrust_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_road_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/ndvi_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/ndwi_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/roughness_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/twi_coarse_25x25.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/slope_25x25.tif"
    ],
    [
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/aspect_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/curvature_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_fault_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_river_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_thrust_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_road_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/ndvi_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/ndwi_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/roughness_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/twi_intermediate_15x15.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/slope_15x15.tif"
    ],
    [
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/aspect_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/curvature_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/dis_to_fault_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/dis_to_river_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/dis_to_thrust_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/dis_to_road_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/ndvi_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/ndwi_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/roughness_3x3.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/twi_finescale.tif",
    "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/slope_3x3.tif"
    ]]
raster_sets = {
    "25x25": raster_files[0],
    "15x15": raster_files[1],
    "3x3": raster_files[2],
}

PATCH_SIZE = 15        # IMPORTANT: same everywhere
HALF = PATCH_SIZE // 2
EPOCHS = 5
BATCH_SIZE = 32
normalize_mode  = "zscore"
df = pd.read_excel(points_xlsx)[["X", "Y", "label"]]

In [4]:
def load_raster_stack(raster_paths):
    """Read rasters, assert same shape/transform/crs, return (H, W, C) float32 stack and profile."""
    # Open first raster to set reference
    with rasterio.open(raster_paths[0]) as ref:
        print(raster_paths[0])
        H, W = ref.height, ref.width
        transform, crs = ref.transform, ref.crs
        profile = ref.profile.copy()           # <- copy while open
        profile.update(count=1, dtype="float32")
        C = len(raster_paths)
        stack = np.zeros((H, W, C), dtype=np.float32)
        stack[..., 0] = ref.read(1).astype(np.float32)
    for i, p in enumerate(raster_paths[1:], start=1):
      print(i)
      with rasterio.open(p) as ds:
          print(raster_paths[i])
          print(f"Height: {ds.height} Width: {ds.width} Transform: {ds.transform}")
          # assert ds.height == H and ds.width == W, f"Shape mismatch: {p}"
          # assert ds.transform == transform,       f"Transform mismatch: {p}"
          assert str(ds.crs) == str(crs),         f"CRS mismatch: {p}"
          # stack[..., i] = ds.read(1).astype(np.float32)
    # Load remaining rasters one-by-one and check consistency
    for i, p in enumerate(raster_paths[1:], start=1):
        print(i)
        with rasterio.open(p) as ds:
            assert ds.height == H and ds.width == W, f"Shape mismatch: {p}"
            assert ds.transform == transform,       f"Transform mismatch: {p}"
            assert str(ds.crs) == str(crs),         f"CRS mismatch: {p}"
            stack[..., i] = ds.read(1).astype(np.float32)

    return stack, transform, crs, profile


In [5]:
def fill_nan_raster(arr):
    mask = np.isnan(arr)
    if not mask.any():
        return arr
    tmp = arr.copy()
    tmp[mask] = 0.0
    _, idx = distance_transform_edt(mask, return_indices=True)
    return tmp[tuple(idx)]



In [6]:
def extract_training_patches(df, stack, transform):
    X, y = [], []
    i=0
    H, W, _ = stack.shape
    for _, row in df.iterrows():
        r, c = rasterio.transform.rowcol(transform, row["X"], row["Y"])

        if r < HALF or c < HALF or r >= H-HALF or c >= W-HALF:
            continue

        patch = stack[
            r-HALF:r+HALF+1,
            c-HALF:c+HALF+1,
            :
        ]

        if np.isnan(patch).any():
            i=i+1
            continue

        X.append(patch)
        y.append(row["label"])
    print("Total Unused Samples: ",i)
    return np.stack(X).astype(np.float32), np.array(y)


In [7]:
def compute_channel_norms(X, mode="zscore"):
    """Compute per-channel normalization stats on training set."""
    # X shape: (N, k, k, C)
    if mode == "zscore":
        means = X.mean(axis=(0,1,2))
        stds  = X.std(axis=(0,1,2)) + 1e-6
        return ("zscore", means, stds)
    elif mode == "minmax":
        mins = X.min(axis=(0,1,2))
        maxs = X.max(axis=(0,1,2))
        return ("minmax", mins, maxs)
    else:
        return (None, None, None)

In [8]:
def apply_channel_norms(X, norms):
    mode, a, b = norms
    if mode == "zscore":
        return (X - a) / b
    elif mode == "minmax":
        return (X - a) / (b - a + 1e-6)
    else:
        return X

In [9]:
def build_cnn(input_shape):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs
    # Block 1
    x = tf.keras.layers.Conv2D(32, (3,3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)

    # Block 2
    x = tf.keras.layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)

    # Block 3
    x = tf.keras.layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)  # robust for variable k (or use Flatten if you keep k fixed)

    # Dense head
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="AUC"), tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model

In [10]:
def predict_full_map(stack, model, norms, k, tile_stride=1, batch=2048):
    """Return probability map (H,W) using sliding window patches (stride = tile_stride)."""
    H, W, C = stack.shape
    half = k // 2

    pad_stack = np.pad(stack, ((half, half), (half, half), (0,0)), mode="reflect")

    centers = [(r, c) for r in range(H) for c in range(W)
               if (r % tile_stride == 0 and c % tile_stride == 0)]
    probs = np.zeros((H, W), dtype=np.float32)

    patches = []
    idxs = []

    # ✅ progress bar here
    for (r, c) in tqdm(centers, total=len(centers), desc="Predicting tiles"):
        r0, c0 = r, c
        pr = r0 + half
        pc = c0 + half
        patch = pad_stack[pr-half:pr+half+1, pc-half:pc+half+1, :]
        patches.append(patch)
        idxs.append((r0, c0))

        if len(patches) == batch:
            Xb = np.stack(patches, axis=0).astype(np.float32)
            Xb = apply_channel_norms(Xb, norms)
            pb = model.predict(Xb, verbose=0).reshape(-1)
            for (rr, cc), p in zip(idxs, pb):
                probs[rr, cc] = p
            patches, idxs = [], []

    if patches:
        Xb = np.stack(patches, axis=0).astype(np.float32)
        Xb = apply_channel_norms(Xb, norms)
        pb = model.predict(Xb, verbose=0).reshape(-1)
        for (rr, cc), p in zip(idxs, pb):
            probs[rr, cc] = p
    print("Prediction batch stats:", Xb.min(), Xb.max(), Xb.mean())
    return probs


In [ ]:
def run_cnn_for_scale(scale_name, raster_list):
    print(f"\n===== Running CNN for scale {scale_name} =====")

    # --- Load rasters ---
    print("Load rasters")
    stack, transform, crs, out_profile = load_raster_stack(raster_list)
    print("Stack shape:", stack.shape)

    # --- Fill NaNs ---
    print("Fill Nans")
    for b in range(stack.shape[-1]):
        stack[..., b] = fill_nan_raster(stack[..., b])
    print("NaNs after fill:", np.isnan(stack).sum())

    # --- Extract patches ---
    print("Extract patches")
    X, y = extract_training_patches(df, stack, transform)
    print("Training samples:", X.shape)

    # --- Train/test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # --- Normalization ---
    print("normalization")
    norms = compute_channel_norms(X_train, normalize_mode)
    X_train = apply_channel_norms(X_train, norms)
    X_test  = apply_channel_norms(X_test, norms)

    # --- Class weights ---
    classes = np.array([0,1])
    cw = compute_class_weight("balanced", classes=classes, y=y_train)
    class_weight_dict = {0: cw[0], 1: cw[1]}
    print("Class weights:", class_weight_dict)

    # --- Build + train CNN ---
    model = build_cnn((PATCH_SIZE, PATCH_SIZE, stack.shape[-1]))
    model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        verbose=1
    )

    # --- Evaluation ---
    y_prob = model.predict(X_test).ravel()
    y_pred = (y_prob > 0.5).astype(int)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    # --- Full map prediction ---
    prob_map = predict_full_map(
        stack, model, norms,
        k=PATCH_SIZE,
        tile_stride=1,
        batch=1024
    )

    # --- Save map ---
    out_profile.update(count=1, dtype="float32")
    out_path = f"/content/drive/MyDrive/CNN_LSM_{scale_name}.tif"

    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(prob_map.astype(np.float32), 1)

    print(f"Saved susceptibility map: {out_path}")
    print("Map stats:", np.nanmin(prob_map), np.nanmax(prob_map))


In [12]:
# for scale, rasters in raster_sets.items():
#     run_cnn_for_scale(scale, rasters)

raster_sets = {
    "25x25": raster_files[0],
    "15x15": raster_files[1],
    "3x3": raster_files[2],
}

In [13]:
run_cnn_for_scale("25x25", raster_files[0])


===== Running CNN for scale 25x25 =====
Load rasters
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/aspect_25x25.tif
1
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/curvature_25x25.tif
Height: 1506 Width: 2290 Transform: | 25.00, 0.00, 544509.88|
| 0.00,-24.99, 3252833.25|
| 0.00, 0.00, 1.00|
2
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_fault_25x25.tif
Height: 1506 Width: 2290 Transform: | 25.00, 0.00, 544509.88|
| 0.00,-24.99, 3252833.25|
| 0.00, 0.00, 1.00|
3
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_river_25x25.tif
Height: 1506 Width: 2290 Transform: | 25.00, 0.00, 544509.88|
| 0.00,-24.99, 3252833.25|
| 0.00, 0.00, 1.00|
4
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/25x25_rasters/dis_to_thrust_25x25.tif
Height: 1506 Width: 2290 Transform: | 25.00,

Predicting tiles: 100%|██████████| 3448740/3448740 [08:56<00:00, 6427.35it/s]


Prediction batch stats: -6.5311093 7.569482 0.2650645
Saved susceptibility map: /content/drive/MyDrive/CNN_LSM_25x25.tif
Map stats: 0.00044567318 0.999815




1.   Epoch vs accuracy
2.   EPoch VS recall
3. Epoch vs



In [14]:
run_cnn_for_scale("15x15", raster_files[1])


===== Running CNN for scale 15x15 =====
Load rasters
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/aspect_15x15.tif
1
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/curvature_15x15.tif
Height: 2509 Width: 3817 Transform: | 15.00, 0.00, 544509.88|
| 0.00,-15.00, 3252833.25|
| 0.00, 0.00, 1.00|
2
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_fault_15x15.tif
Height: 2509 Width: 3817 Transform: | 15.00, 0.00, 544509.88|
| 0.00,-15.00, 3252833.25|
| 0.00, 0.00, 1.00|
3
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_river_15x15.tif
Height: 2509 Width: 3817 Transform: | 15.00, 0.00, 544509.88|
| 0.00,-15.00, 3252833.25|
| 0.00, 0.00, 1.00|
4
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/15x15_rasters/dis_to_thrust_15x15.tif
Height: 2509 Width: 3817 Transform: | 15.00,

Predicting tiles: 100%|██████████| 9576853/9576853 [25:37<00:00, 6229.39it/s]


Prediction batch stats: -6.3097515 5.0947633 0.40039605
Saved susceptibility map: /content/drive/MyDrive/CNN_LSM_15x15.tif
Map stats: 2.311323e-05 0.9999727


In [ ]:
run_cnn_for_scale("3x3", raster_files[2])


===== Running CNN for scale 3x3 =====
Load rasters
/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/3x3_rasters/aspect_3x3.tif
